In [1]:
import torch
import torch.nn as nn
from gensim.models import Word2Vec
import numpy as np
from sklearn.model_selection import train_test_split
import re
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import torch.nn.functional as F
import difflib

In [2]:
# Dictionary mapping contractions to their full forms
contractions_dict = {
    "he's": "he is",
    "i'm": "I am",
    "you're": "you are",
    "we've": "we have",
    "they've": "they have",
    "don't": "do not",
    "isn't": "is not",
    "it's": "it is",
    "didn't": "did not",
    "aren't": "are not",
    "let's": "let us",
    "couldn't": "could not",
    "wasn't": "was not",
    "weren't": "were not",
    "ain't": "am not",
    "i've": "I have",
    "that's": "that is",
    "i'll": "I will",
    "you'd": "you would",
    "they're": "they are",
    "i won't": "I will not",
    "can't": "cannot",
    "you've": "you have",
    "there's": "there is",
    "won't": "will not",
    "you'll": "you will",
    "doesn't": "does not",
    "must've": "must have",
    "what's": "what is",
    "we're": "we are",
    "haven't": "have not",
    "wouldn't": "would not",
    "i'd": "I would",
    "she's": "she is",
    "nobody's": "nobody is",
    "we'll": "we will",
    "they'd": "they would",
    "mustn't": "must not",
    "could've": "could have",
    "shouldn't": "should not",
    "he'll": "he will",
    "he'd": "he would",
    "hadn't": "had not",
    "where'd": "where did",
    "we'd": "we would",
}

# Function to replace contractions in the text
def replace_contractions(text, contractions_map):
    # Create a regex pattern that matches any of the contractions
    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in contractions_map.keys()) + r')\b')
    # Replace contractions using the dictionary
    return pattern.sub(lambda x: contractions_map[x.group()], text)

# Function to process the text file
def process_text_file(input_file, output_file, contractions_map):
    # Read the contents of the input file
    with open(input_file, 'r') as file:
        text = file.read().lower()

    # Replace contractions
    new_text = replace_contractions(text, contractions_map)

    # Write the modified text to the output file
    with open(output_file, 'w') as file:
        file.write(new_text)

    return new_text

# Specify the input and output file paths
input_file = 'adele.txt'  # Replace with the actual file path
output_file = 'output.txt'

# Process the text file
modified_texte = process_text_file(input_file, output_file, contractions_dict)

print("Contractions replaced and saved to", output_file)

Contractions replaced and saved to output.txt


# Parameters

In [3]:
# It is arbitrary values
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
N_LAYERS = 2
DROPOUT = 0.5
N_EPOCHS = 100
LR = 3e-3
BATCH_SIZE = 32
SEQ_LEN = 30

# Tokenization

In [4]:
with open("adele.txt", "r") as f:
    sentences = f.readlines()

text = modified_texte

sentences_with_EOS = []
for sentence in sentences:
    sentence_contraction = sentence.lower()
    sentence_contraction = replace_contractions(sentence_contraction, contractions_dict)
    # remove non alpha numerique character
    sentence_contraction = re.sub(r'[^a-zA-Z\s\<\>]', '', sentence_contraction)
    sentence_contraction = sentence_contraction.split()
    if len(sentence_contraction) > 0:
        sentence_contraction.append("<EOS>")
        sentences_with_EOS.append(sentence_contraction)
    else: 
        print("phrase ignored: ", sentence_contraction) 

sentences = sentences_with_EOS
print(sentences)

word2vec_model = Word2Vec(sentences, vector_size=EMBEDDING_DIM, window=5, min_count=1, epochs=100)
word2vec_model.save("word2vec100_adele.model")


phrase ignored:  []
phrase ignored:  []
[['looking', 'for', 'some', 'education', '<EOS>'], ['made', 'my', 'way', 'into', 'the', 'night', '<EOS>'], ['all', 'that', 'bullshit', 'conversation', '<EOS>'], ['baby', 'cannot', 'you', 'read', 'the', 'signs', 'I', 'will', 'not', 'bore', 'you', 'with', 'the', 'details', 'baby', '<EOS>'], ['i', 'do', 'not', 'even', 'wanna', 'waste', 'your', 'time', '<EOS>'], ['let', 'us', 'just', 'say', 'that', 'maybe', '<EOS>'], ['you', 'could', 'help', 'me', 'ease', 'my', 'mind', '<EOS>'], ['i', 'am', 'not', 'mr', 'right', 'but', 'if', 'you', 'are', 'looking', 'for', 'fast', 'love', '<EOS>'], ['if', 'that', 'is', 'love', 'in', 'your', 'eyes', '<EOS>'], ['it', 'is', 'more', 'than', 'enough', '<EOS>'], ['had', 'some', 'bad', 'love', '<EOS>'], ['so', 'fast', 'love', 'is', 'all', 'that', 'I', 'have', 'got', 'on', 'my', 'mind', 'ooh', 'ooh', '<EOS>'], ['ooh', 'ooh', 'looking', 'for', 'some', 'affirmation', '<EOS>'], ['made', 'my', 'way', 'into', 'the', 'sun', '<EOS>

In [5]:
split_index = int(len(sentences) * 0.8)
train_sentences = sentences[:split_index]
val_sentences = sentences[split_index:]

print(f"Number of training sentences: {len(train_sentences)}")
print(f"Number of validation sentences: {len(val_sentences)}")

Number of training sentences: 1918
Number of validation sentences: 480


In [6]:
# Create word-to-index and index-to-word mappings
word2idx = {word: idx for idx, word in enumerate(word2vec_model.wv.index_to_key)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 1344


In [7]:
# Initialize embedding matrix with zeros for the vocabulary
embedding_dim = word2vec_model.vector_size  # Word2Vec embedding size
embedding_matrix = np.zeros((vocab_size, embedding_dim), dtype=np.float32)  # Ensure it's a 2D matrix

for word, idx in word2idx.items():
    if word in word2vec_model.wv:
        embedding_matrix[idx] = word2vec_model.wv[word]  # Use Word2Vec vector
    else:
        embedding_matrix[idx] = np.random.normal(size=(embedding_dim,))  # Random initialization for OOV words

# Convert to PyTorch tensor
embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float)

print(embedding_matrix)

tensor([[-0.0153, -0.3430,  0.1301,  ..., -0.0814, -0.5844, -0.3016],
        [-0.5427, -0.2988, -0.2689,  ...,  0.0154,  0.2493,  0.4257],
        [-0.8124, -0.6676, -1.1641,  ...,  0.0088,  0.6629, -0.7179],
        ...,
        [-0.0121,  0.0583,  0.0605,  ..., -0.2032,  0.0691, -0.0437],
        [-0.2168,  0.0601,  0.1129,  ...,  0.0350,  0.0958, -0.3480],
        [ 0.0366,  0.0847, -0.0637,  ...,  0.0847, -0.0351,  0.0238]])


In [8]:
class IndexedTextDataset(Dataset):
    def __init__(self, sentences, word2idx, max_len=None):
        """
        Dataset class that returns word indices for sentences.

        Args:
        - sentences (list of list of str): The sentences, each represented as a list of words.
        - word2idx (dict): Mapping from words to indices.
        - max_len (int, optional): Maximum sentence length. Sentences longer than this are truncated.
        """
        self.sentences = sentences
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        
        # Convert each word in the sentence to its index
        indexed_sentence = [self.word2idx.get(word) for word in sentence]  # Use None for unknown words
        
        # Optionally truncate to max_len if specified
        if self.max_len:
            indexed_sentence = indexed_sentence[:self.max_len]
        
        # Create input-output pairs (shifted by one position)
        X = indexed_sentence[:-1]
        y = indexed_sentence[1:]
        
        return X, y


def collate_fn(batch):
    # Separate X and y for all samples in the batch
    X_batch, y_batch = zip(*batch)
    
    # Convert X and y to tensors, applying padding
    X_padded = pad_sequence([torch.tensor(x) for x in X_batch], batch_first=True, padding_value=0)  # Use padding_value=0
    y_padded = pad_sequence([torch.tensor(y) for y in y_batch], batch_first=True, padding_value=0)
    
    # Record the original lengths of each sequence for dynamic processing
    X_lengths = torch.tensor([len(x) for x in X_batch])

    return X_padded, y_padded, X_lengths

# Create train and validation datasets
train_dataset = IndexedTextDataset(train_sentences, word2idx)
val_dataset = IndexedTextDataset(val_sentences, word2idx)

# Create DataLoader with dynamic padding
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# Print some examples
for X, y, X_lengths in train_loader:
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("X lengths:", X_lengths)
    break

X shape: torch.Size([32, 21])
y shape: torch.Size([32, 21])
X lengths: tensor([18,  7,  4, 15,  6,  5, 17, 19,  6, 13, 10, 10,  9,  5, 14,  7,  9,  4,
         6, 11,  5,  4,  8, 12, 11,  4,  6,  9, 11,  6, 21,  9])


# Model

In [9]:
class Word2VecNLP(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix, hidden_dim, n_layers, dropout, model_type='LSTM'):
        super(Word2VecNLP, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = n_layers
        self.rnn_type = model_type
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix)
        
        # Initialize LSTM or GRU layer with embedding dimension as input size
        if model_type == 'LSTM':
            self.nlp = nn.LSTM(embedding_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        elif model_type == 'GRU':
            self.nlp = nn.GRU(embedding_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        else:
            raise Exception("Model type not supported")

        # Output layer that maps hidden states to embedding space
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, lengths, hidden):
        # Unpack inputs into x and lengths for packed sequence
        x = self.embedding(x)
        # Pack padded sequences for efficient processing
        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        
        # Forward pass through LSTM or GRU
        packed_output, hidden = self.nlp(packed_input, hidden)
        
        # Unpack the sequence
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        
        # Map the LSTM/GRU output to the embedding space
        output = self.fc(output)
        
        return output, hidden

    def init_hidden(self, batch_size):
        # Initialize hidden state for LSTM or GRU
        if self.rnn_type == 'LSTM':
            # LSTM requires both hidden state and cell state
            hidden = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            cell = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            return (hidden, cell)
        else:
            # GRU only requires the hidden state
            hidden = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            return hidden


In [10]:
def train(model, dataloader, n_epochs, lr, batch_size, name):
    # Setup GPU related variables
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device = {device}")
    torch.cuda.empty_cache()
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction='mean')

    model.train()
    
    for epoch in range(n_epochs):
        train_losses = []
        for i, (X_padded, y_padded, X_lengths) in enumerate(dataloader):
            X_padded, y_padded = X_padded.to(device), y_padded.to(device)
            X_lengths = X_lengths.to(device)
            
            # Initialize hidden state
            hidden = model.init_hidden(batch_size)

            # Forward pass through the model
            optimizer.zero_grad()
            output, hidden = model(X_padded, X_lengths, hidden)

            #print(output.shape)
            output = output.view(-1, vocab_size)
            y_padded = y_padded.view(-1)
            
            loss = criterion(output, y_padded.to(model.device))
            
            # Backpropagation and optimization
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            
            if i % 10 == 0:
                print(f"Epoch {epoch}, step {i}, loss {loss.item()}")
        
        print(f"Epoch {epoch} finished. Train loss: {np.array(train_losses).mean()}, Perplexity: {np.exp(np.array(train_losses).mean())}")

    torch.save(model.state_dict(), f"model_save/{name}.pth")

In [11]:
model_type = "GRU"
GRU_model = Word2VecNLP(vocab_size, EMBEDDING_DIM, embedding_matrix, HIDDEN_DIM, N_LAYERS, DROPOUT, model_type)

# train(GRU_model, train_vectors, train_sentences, N_EPOCHS, LR, BATCH_SIZE, SEQ_LEN, f"{model_type}_model_Word2Vec{EMBEDDING_DIM}")
train(GRU_model, train_loader, N_EPOCHS, LR, BATCH_SIZE, f"{model_type}_model_Word2Vec{EMBEDDING_DIM}")


device = cpu
Epoch 0, step 0, loss 7.170284748077393
Epoch 0, step 10, loss 6.467569828033447
Epoch 0, step 20, loss 6.469130992889404
Epoch 0, step 30, loss 6.125243186950684
Epoch 0, step 40, loss 5.87462043762207
Epoch 0, step 50, loss 6.0188469886779785
Epoch 0 finished. Train loss: 6.270697069168091, Perplexity: 528.8458915886906
Epoch 1, step 0, loss 5.590028762817383
Epoch 1, step 10, loss 5.636325836181641
Epoch 1, step 20, loss 5.58179235458374
Epoch 1, step 30, loss 5.790081024169922
Epoch 1, step 40, loss 5.395644664764404
Epoch 1, step 50, loss 5.1304144859313965
Epoch 1 finished. Train loss: 5.533843795458476, Perplexity: 253.11496571580375
Epoch 2, step 0, loss 5.179177284240723
Epoch 2, step 10, loss 4.91376256942749
Epoch 2, step 20, loss 4.604040622711182
Epoch 2, step 30, loss 4.811570644378662
Epoch 2, step 40, loss 4.737051010131836
Epoch 2, step 50, loss 4.684007167816162
Epoch 2 finished. Train loss: 4.960840439796447, Perplexity: 142.71368777168178
Epoch 3, step 

In [12]:
model_type = "LSTM"
LSTM_model = Word2VecNLP(vocab_size, EMBEDDING_DIM, embedding_matrix, HIDDEN_DIM, N_LAYERS, DROPOUT, model_type)

# train(LSTM_model, train_vectors, train_sentences, N_EPOCHS, LR, BATCH_SIZE, SEQ_LEN, f"{model_type}_model_Word2Vec{EMBEDDING_DIM}")
train(LSTM_model, train_loader, N_EPOCHS, LR, BATCH_SIZE, f"{model_type}_model_Word2Vec{EMBEDDING_DIM}")

device = cpu
Epoch 0, step 0, loss 7.190926551818848
Epoch 0, step 10, loss 6.48462438583374
Epoch 0, step 20, loss 6.240678310394287
Epoch 0, step 30, loss 6.124201774597168
Epoch 0, step 40, loss 6.356753826141357
Epoch 0, step 50, loss 6.2364888191223145
Epoch 0 finished. Train loss: 6.364213045438131, Perplexity: 580.6876736468415
Epoch 1, step 0, loss 6.165886878967285
Epoch 1, step 10, loss 5.944401264190674
Epoch 1, step 20, loss 6.1428961753845215
Epoch 1, step 30, loss 5.967342376708984
Epoch 1, step 40, loss 5.635009765625
Epoch 1, step 50, loss 5.80427360534668
Epoch 1 finished. Train loss: 5.934339833259583, Perplexity: 377.79050910021795
Epoch 2, step 0, loss 5.639176845550537
Epoch 2, step 10, loss 5.533724784851074
Epoch 2, step 20, loss 5.640199184417725
Epoch 2, step 30, loss 5.320517539978027
Epoch 2, step 40, loss 5.515786647796631
Epoch 2, step 50, loss 5.70266580581665
Epoch 2 finished. Train loss: 5.5180064757665, Perplexity: 249.13787940621788
Epoch 3, step 0, lo

# Evaluation of the model

In [13]:
GRU_model = Word2VecNLP(vocab_size, EMBEDDING_DIM, embedding_matrix, HIDDEN_DIM, N_LAYERS, DROPOUT, 'GRU')
GRU_model.load_state_dict(torch.load("model_save/GRU_model_Word2Vec100.pth", weights_only=True))

LSTM_model = Word2VecNLP(vocab_size, EMBEDDING_DIM, embedding_matrix, HIDDEN_DIM, N_LAYERS, DROPOUT, 'LSTM')
LSTM_model.load_state_dict(torch.load("model_save/LSTM_model_Word2Vec100.pth", weights_only=True))

<All keys matched successfully>

In [14]:
def computer_word_similarity(reference, prediction):
    """
    Function to compute the similarity between the reference text and the
    predicted text (similarity between the 2 sequences at the word level).

    Args:
        reference: Reference text
        prediction: Predicted text

    Returns:
        Similarity ratio between words
    """
    ref_words = reference.split()
    pred_words = prediction.split()

    matcher = difflib.SequenceMatcher(None, ref_words, pred_words)

    ratio = matcher.ratio()

    return ratio

In [15]:
def generate_text_with_metrics(model, word2idx, idx2word, start_word, target_text, num_words=10, random_sample=False, top_k=10):
    """
    Generate text based on the trained model output and compute evaluation metrics.
    
    Parameters:
    - model: The trained PyTorch model.
    - word2idx: Dictionary mapping words to their indices.
    - idx2word: Dictionary mapping indices to their words.
    - start_word: The initial word to start generating text.
    - target_text: The target text to compare against.
    - num_words: Number of words to generate.
    - random_sample: If True, sample from the distribution instead of taking the max probability.
    - top_k: Top-k accuracy to compute.
    
    Returns:
    - metrics: A dictionary containing accuracy, top-k accuracy, F1 score, and perplexity.
    - generated_text: The generated sequence of words.
    """
    model.eval()
    
    start_word = start_word.lower().split()
    start_word_size = len(start_word)
    generated_words = start_word
    
    # Convert the start word to its index
    input_idx = [word2idx.get(word) for word in start_word]
    input_tensor = torch.tensor([input_idx], dtype=torch.long).to(model.device)
    
    # Initialize hidden state
    hidden = model.init_hidden(batch_size=1)
    log_probs = []
    predictions = []
    top_k_acc = 0
    
    # Generate the specified number of words
    for i in range(num_words):
        with torch.no_grad():
            output, hidden = model(input_tensor, torch.tensor([1]), hidden)
            # Get the output probabilities for the last word in the sequence
            output = output.squeeze(0)  # Remove batch dimension: [seq_len, vocab_size]
            probabilities = F.softmax(output[-1], dim=0).cpu().numpy()  # Get probabilities for last time step

            # Get the top-k indices
            top_k_indices = np.argsort(-probabilities)[:top_k][::-1]
            
            # Check if the target word is in the top-k predictions
            for idx in top_k_indices:
                top_k_encoding = np.zeros_like(probabilities)
                top_k_encoding[idx] = 1
                top_k_next_word = idx2word[top_k_encoding.argmax()]
                
                if i + start_word_size < len(target_text):
                    target_word = target_text[i + start_word_size]
                    if top_k_next_word == target_word:
                        top_k_acc += 1
                        break
        
        # Choose the next word based on probabilities or highest probability
        if random_sample:
            next_idx = np.random.choice(len(probabilities), p=probabilities)
        else:
            next_idx = np.argmax(probabilities)

        next_word = idx2word[next_idx]

        # Stop generation if end-of-sequence token is reached
        if next_word == "<EOS>":
            #print("End-of-sequence token <EOS> reached.")
            break

        # Append the selected word to the generated sequence
        generated_words.append(next_word)
        predictions.append(next_word)
        log_probs.append(np.log(probabilities[next_idx]))

        # Update input with the new word index for the next iteration
        input_tensor = torch.tensor([[next_idx]], dtype=torch.long).to(model.device)

    # Calculate evaluation metrics
    targets = target_text[start_word_size:]
    accuracy = 0
    
    for i in range(len(predictions)):
        if i < len(targets) and predictions[i] == targets[i]:
            accuracy += 1
    accuracy /= len(targets)
    
    top_k_acc /= len(targets)
    
    perplexity = np.exp(-np.mean(log_probs)) if log_probs else float('inf')
    
    metrics = {
        "accuracy": accuracy,
        "top_k_accuracy": top_k_acc,
        "perplexity": perplexity,
        'ratio': computer_word_similarity(' '.join(targets), ' '.join(predictions))
    }
    
    # Join the generated words into a single string
    generated_text = ' '.join(generated_words)
    return metrics, generated_text


In [16]:
testing_sentences = []
sentence = []
for sentence in val_sentences:
    if len(sentence) >= 3:
        testing_sentences.append(sentence)

In [17]:
def evaluate(testing_sentences, model, word2idx, idx2word):
    # Compute the metrics for all testing sequences
    results = []
    for sentences in testing_sentences:
        metrics, _ = generate_text_with_metrics(model, word2idx, idx2word, " ".join(sentences[0:2]), sentences ,num_words=len(sentences)-2, random_sample=False)
        results.append(metrics)

    # Aggreate results into a single dictonary
    res = {}
    for key in results[0].keys():
        acc = []
        for result in results:
            if result[key] == float('inf'):
                continue
            acc.append(result[key])
        res[key] = np.array(acc).mean()

    return res
print(f"Metrics for the GRU Model: {evaluate(testing_sentences, GRU_model, word2idx, idx2word)}")
print(f"Metrics for the LSTM Model: {evaluate(testing_sentences, LSTM_model, word2idx, idx2word)}")

Metrics for the GRU Model: {'accuracy': 0.02634289695728288, 'top_k_accuracy': 0.18126105450216168, 'perplexity': 1.7024717, 'ratio': 0.08602641314541766}
Metrics for the LSTM Model: {'accuracy': 0.015311014804923247, 'top_k_accuracy': 0.1348847380110739, 'perplexity': 1.8404915, 'ratio': 0.07071522961616199}


# Testing the GRU model on the test set

In [25]:
metrics, generated_text = generate_text_with_metrics(GRU_model, word2idx, idx2word, "you will never see", testing_sentences[2], num_words=10, random_sample=False)
print(f"Target: {' '.join(testing_sentences[2])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

metrics, generated_text = generate_text_with_metrics(GRU_model, word2idx, idx2word, "let the", testing_sentences[10], num_words=10, random_sample=False)
print(f"Target: {' '.join(testing_sentences[10])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

metrics, generated_text = generate_text_with_metrics(GRU_model, word2idx, idx2word, "is all of", testing_sentences[20], num_words=10, random_sample=False)
print(f"Target: {' '.join(testing_sentences[20])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

Target: seems i love the things you do <EOS>
Generated text: you will never see are gonna wish you never had met me
Metrics values of the generated text: {'accuracy': 0.0, 'top_k_accuracy': 0.0, 'perplexity': 1.3502922, 'ratio': 0.16666666666666666}

Target: it was dark and i was over <EOS>
Generated text: let the the weight take me under
Metrics values of the generated text: {'accuracy': 0.0, 'top_k_accuracy': 0.16666666666666666, 'perplexity': 1.3483431, 'ratio': 0.0}

Target: watched it pour as i touched your face <EOS>
Generated text: is all of there anyone i could call
Metrics values of the generated text: {'accuracy': 0.0, 'top_k_accuracy': 0.16666666666666666, 'perplexity': 1.5687371, 'ratio': 0.18181818181818182}



In [ ]:
metrics, generated_text = generate_text_with_metrics(LSTM_model, word2idx, idx2word, "you will never see", testing_sentences[2], num_words=10, random_sample=False)
print(f"Target: {' '.join(testing_sentences[2])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

metrics, generated_text = generate_text_with_metrics(LSTM_model, word2idx, idx2word, "let the", testing_sentences[10], num_words=10, random_sample=False)
print(f"Target: {' '.join(testing_sentences[10])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

metrics, generated_text = generate_text_with_metrics(LSTM_model, word2idx, idx2word, "is all of", testing_sentences[20], num_words=10, random_sample=False)
print(f"Target: {' '.join(testing_sentences[20])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

['seems', 'i', 'love', 'the', 'things', 'you', 'do', '<EOS>']
{'accuracy': 0.0, 'top_k_accuracy': 0.0, 'perplexity': 1.2755718, 'ratio': 0.16666666666666666}
you will never see are gonna wish you never had met me

['it', 'was', 'dark', 'and', 'i', 'was', 'over', '<EOS>']
{'accuracy': 0.0, 'top_k_accuracy': 0.16666666666666666, 'perplexity': 1.3179961, 'ratio': 0.0}
let the the sky fall when it crumbles

['watched', 'it', 'pour', 'as', 'i', 'touched', 'your', 'face', '<EOS>']
{'accuracy': 0.0, 'top_k_accuracy': 0.3333333333333333, 'perplexity': 1.1993839, 'ratio': 0.18181818181818182}
is all of there anyone i could call



In [20]:
def prepare_input_sentence(sentence, contractions_dict, word2idx):
    """
    Preprocess the input sentence to match the vocabulary requirements of the model.
    
    Parameters:
    - sentence: str, the input sentence to process.
    - contractions_dict: dict, mapping of contractions to their expanded forms.
    - word2idx: dict, mapping of words to their indices in the vocabulary.
    
    Returns:
    - processed_indices: List of indices for each word in the sentence, ready for model input.
    """
    # Step 1: Replace contractions
    sentence = replace_contractions(sentence.lower(), contractions_dict)

    # Step 2: Remove non-alphanumeric characters
    sentence = re.sub(r'[^\w\s]', '', sentence)
    
    # Step 3: translate to lower case
    sentence = sentence.lower()
    
    # Step 4: Tokenize the sentence
    words = sentence.split()
    
    # Step 5: Convert words to indices, using <UNK> for unknown words
    processed_indices = [word2idx.get(word) for word in words]
    
    return processed_indices


def generate_text(model, word2idx, idx2word, start_sentence, num_words=10, random_sample=False):
    """
    Generate text based on the trained model output.

    Parameters:
    - model: The trained PyTorch model.
    - word2idx: Dictionary mapping words to their indices.
    - idx2word: Dictionary mapping indices to words.
    - start_word: The initial word to start generating text.
    - num_words: Number of words to generate.
    - random_sample: If True, sample from the distribution instead of taking the max probability.

    Returns:
    - generated_text: The generated sequence of words.
    """
    model.eval()
    
    start_indices = prepare_input_sentence(start_sentence, contractions_dict, word2idx)
    if not start_indices:  # If the sentence is empty after preprocessing, return early
        return "<No valid input>"
    
    # Initialize the generated text with the start word
    generated_words = [idx2word[idx] for idx in start_indices if idx in idx2word]
    
    # Convert the start word to its index
    input_idx = torch.tensor([start_indices], dtype=torch.long).to(model.device)

    # Initialize hidden state
    hidden = model.init_hidden(batch_size=1)

    # Generate the specified number of words
    for _ in range(num_words):
        with torch.no_grad():
            output, hidden = model(input_idx, torch.tensor([1]), hidden)  # Pass sequence length of 1

        # Get the output probabilities for the last word in the sequence
        output = output.squeeze(0)  # Remove batch dimension: [seq_len, vocab_size]
        probabilities = F.softmax(output[-1], dim=0).cpu().numpy()  # Get probabilities for last time step

        # Choose the next word based on probabilities or highest probability
        if random_sample:
            next_idx = np.random.choice(len(probabilities), p=probabilities)
        else:
            next_idx = np.argmax(probabilities)

        next_word = idx2word[next_idx]

        # Stop generation if end-of-sequence token is reached
        if next_word == "<EOS>":
            print("End-of-sequence token <EOS> reached.")
            break

        # Append the selected word to the generated sequence
        generated_words.append(next_word)

        # Update input with the new word index for the next iteration
        input_idx = torch.tensor([[next_idx]], dtype=torch.long).to(model.device)

    # Join the generated words into a single string
    generated_text = ' '.join(generated_words)
    return generated_text


In [21]:
# Example input sentence
input_sentence = "I'm going to test this model!"

# Prepare input sentence
input_indices = prepare_input_sentence(input_sentence, contractions_dict, word2idx)
print("Processed indices:", input_indices)


Processed indices: [2, 11, 644, 6, None, 45, None]


In [22]:
# Exemple d'utilisation
generated_text = generate_text(GRU_model, word2idx, idx2word, start_sentence='So', num_words=10, random_sample=False)
print(generated_text)

generated_text = generate_text(GRU_model, word2idx, idx2word, start_sentence='I know there is no', num_words=10, random_sample=False)
print(generated_text)

generated_text = generate_text(GRU_model, word2idx, idx2word, start_sentence='I am', num_words=10, random_sample=False)
print(generated_text)

generated_text_random = generate_text(GRU_model, word2idx, idx2word, start_sentence='Baby', num_words=10, random_sample=True)
print(generated_text_random)

so fast love is all that I have got on my
End-of-sequence token <EOS> reached.
i know there is no know i left you speechless
End-of-sequence token <EOS> reached.
i am know i left you speechless
End-of-sequence token <EOS> reached.
baby baby


In [23]:
# Exemple d'utilisation
generated_text = generate_text(LSTM_model, word2idx, idx2word, start_sentence='who i', random_sample=False)
print(generated_text)

generated_text = generate_text(LSTM_model, word2idx, idx2word, start_sentence='had some', random_sample=False)
print(generated_text)

generated_text = generate_text(LSTM_model, word2idx, idx2word, start_sentence='every time he am', random_sample=False)
print(generated_text)

generated_text_random = generate_text(LSTM_model, word2idx, idx2word, start_sentence='Baby', random_sample=True)
print(generated_text_random)

End-of-sequence token <EOS> reached.
who i wants to be right as rain
End-of-sequence token <EOS> reached.
had some some bad love
End-of-sequence token <EOS> reached.
every time he am feeling every word
End-of-sequence token <EOS> reached.
baby baby
